# 북항, 신항 환적 효율성
- `data.csv`가 북항/신항이 환적항으로 얼마나 효율적으로 운영되는가 라는 질문에 답할 수 있는 데이터인가?


In [48]:
import pandas as pd
from PIL.DdsImagePlugin import DXGI_FORMAT_R8G8B8A8_TYPELESS

TARGET_PATH = '../data/data.csv'


In [49]:
df = pd.read_csv(TARGET_PATH, encoding='euc-kr')

In [50]:
df.shape

(2409, 17)

In [51]:
df.head()

,연도,월,분기,청코드,내외항구분,수출입구분명,시설코드,시설명,부두구분명,아외국구분,적공구분,컨테이너수(10피트),컨테이너수(20피트),컨테이너수(40피트),컨테이너수(기타),전체개수,전체물동량
0,2024,3,1,신항,외항,수입,6,신항 W 정박지,일반부두,외국선,적컨,0,3,8,0,11,19.0
1,2024,4,2,신항,외항,수입,6,신항 W 정박지,일반부두,아국선,적컨,0,9,3,0,12,15.0
2,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,공컨,0,1,0,0,1,1.0
3,2024,11,4,신항,외항,수입,7,신항 U 정박지,일반부두,아국선,공컨,0,57,2,0,59,61.0
4,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,적컨,0,31,100,0,131,231.0


In [52]:
df.columns.to_list()

['연도',
 '월',
 '분기',
 '청코드',
 '내외항구분',
 '수출입구분명',
 '시설코드',
 '시설명',
 '부두구분명',
 '아외국구분',
 '적공구분',
 '컨테이너수(10피트)',
 '컨테이너수(20피트)',
 '컨테이너수(40피트)',
 '컨테이너수(기타)',
 '전체개수',
 '전체물동량']

In [53]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2409 entries, 0 to 2408
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   연도           2409 non-null   int64  
 1   월            2409 non-null   int64  
 2   분기           2409 non-null   int64  
 3   청코드          2409 non-null   str    
 4   내외항구분        2409 non-null   str    
 5   수출입구분명       2409 non-null   str    
 6   시설코드         2409 non-null   int64  
 7   시설명          2409 non-null   str    
 8   부두구분명        2409 non-null   str    
 9   아외국구분        2409 non-null   str    
 10  적공구분         2409 non-null   str    
 11  컨테이너수(10피트)  2409 non-null   int64  
 12  컨테이너수(20피트)  2409 non-null   int64  
 13  컨테이너수(40피트)  2409 non-null   int64  
 14  컨테이너수(기타)    2409 non-null   int64  
 15  전체개수         2409 non-null   int64  
 16  전체물동량        2409 non-null   float64
dtypes: float64(1), int64(9), str(7)
memory usage: 320.1 KB


In [54]:
df['연도'].unique()

array([2024])

In [55]:
# 어떤 데이터가 범주형이고, 어떤 데이터가 수치형인지 구분해야함

for col in ['청코드', '내외항구분', '수출입구분명', '아외국구분', '적공구분','부두구분명']:
    print(col, ":", df[col].unique())

청코드 : <StringArray>
['신항', '북항', '감천']
Length: 3, dtype: str
내외항구분 : <StringArray>
['외항']
Length: 1, dtype: str
수출입구분명 : <StringArray>
['수입', '수입환적', '수출환적', '수출']
Length: 4, dtype: str
아외국구분 : <StringArray>
['외국선', '아국선']
Length: 2, dtype: str
적공구분 : <StringArray>
['적컨', '공컨']
Length: 2, dtype: str
부두구분명 : <StringArray>
['일반부두', '컨테이너부두']
Length: 2, dtype: str


In [56]:
df.groupby(['청코드', '월']).size().unstack(level=0)

청코드,감천,북항,신항
월,,,
1,7,87,97
2,9,87,101
3,11,94,98
4,8,77,107
5,13,75,113
6,9,70,110
7,10,79,116
8,8,70,112
9,7,89,114


In [57]:
df['전체물동량'].isna().sum()

np.int64(0)

## 목표
- 환적을 얼마나 많이 하는가(규모)
- 환적을 얼마나 효율적으로 처리하는가(효율)

In [58]:
# 환적 비중 계산 - 환적여부라는 컬럼 생성

df['환적여부'] = df['수출입구분명'].isin(['수입환적','수출환적'])

In [59]:
zone_total = df.groupby('청코드')['전체물동량'].sum() # 항만별 처리한 전체 컨테이너 TEU 처리량
zone_total

청코드
감천        9014.0
북항     6512510.0
신항    17880496.0
Name: 전체물동량, dtype: float64

In [60]:
zone_transship = df[df['환적여부']].groupby('청코드')['전체물동량'].sum() # 항만별 전체 환적 컨테이너 teu 처리량

In [61]:
zone_share = (zone_transship / zone_total * 100).round(2)
zone_share

청코드
감천    68.69
북항    35.54
신항    62.51
Name: 전체물동량, dtype: float64

In [62]:
df.columns

Index(['연도', '월', '분기', '청코드', '내외항구분', '수출입구분명', '시설코드', '시설명', '부두구분명',
       '아외국구분', '적공구분', '컨테이너수(10피트)', '컨테이너수(20피트)', '컨테이너수(40피트)',
       '컨테이너수(기타)', '전체개수', '전체물동량', '환적여부'],
      dtype='str')

In [63]:
df['컨테이너수(기타)']

0       0
1       0
2       0
3       0
4       0
       ..
2404    0
2405    2
2406    0
2407    3
2408    0
Name: 컨테이너수(기타), Length: 2409, dtype: int64

In [64]:
teu_estimate = df['컨테이너수(10피트)'] * 0.5 + df['컨테이너수(20피트)'] * 1 + df['컨테이너수(40피트)'] * 2
teu_estimate

0          19.0
1          15.0
2           1.0
3          61.0
4         231.0
         ...   
2404    21637.0
2405    20595.0
2406    22324.0
2407     6555.0
2408    11877.0
Length: 2409, dtype: float64

In [65]:
grp = df.groupby(['청코드','환적여부']).agg(물동량=('전체물동량','sum'), 개수=('전체개수','sum'))
grp['TEU_FACTOR'] = grp['물동량'] / grp['개수']

grp

물동량       개수  TEU_FACTOR
청코드 환적여부                                   
감천  False      2822.00     1516    1.861478
    True       6192.00     3210    1.928972
북항  False   4197997.25  2859718    1.467976
    True    2314512.75  1569611    1.474577
신항  False   6704016.75  4140278    1.619219
    True   11176479.25  6464959    1.728778

In [96]:
grp2 = df.groupby(['청코드','환적여부','적공구분'])['전체물동량'].sum().unstack(fill_value=0)

In [97]:
grp2['공컨비율(%)'] = ((grp2['공컨'] / (grp2['공컨'] + grp2['적컨'])) * 100).round(2)

grp2

적공구분              공컨           적컨  공컨비율(%)
청코드 환적여부                                  
감천  False     1083.0      1739.00    38.38
    True         0.0      6192.00     0.00
북항  False  1314340.5   2883656.75    31.31
    True     45790.0   2268722.75     1.98
신항  False  1983822.0   4720194.75    29.59
    True    713056.0  10463423.25     6.38

eda 끝/ 시각화

In [75]:
import plotly.express as px

px

<module 'plotly.express' from 'C:\\WORK\\dashboard_con\\.venv\\Lib\\site-packages\\plotly\\express\\__init__.py'>

In [81]:
grp

물동량       개수  TEU_FACTOR
청코드 환적여부                                   
감천  False      2822.00     1516    1.861478
    True       6192.00     3210    1.928972
북항  False   4197997.25  2859718    1.467976
    True    2314512.75  1569611    1.474577
신항  False   6704016.75  4140278    1.619219
    True   11176479.25  6464959    1.728778

In [80]:
grp_plot = grp.reset_index() #수정된 인덱스를 초기화해줌
grp_plot['환적여부'] = grp_plot['환적여부'].map({True: '환적', False: '수출입(일반)'})

grp_plot

,청코드,환적여부,물동량,개수,TEU_FACTOR
0,감천,수출입(일반),2822.00,1516,1.861478
1,감천,환적,6192.00,3210,1.928972
2,북항,수출입(일반),4197997.25,2859718,1.467976
3,북항,환적,2314512.75,1569611,1.474577
4,신항,수출입(일반),6704016.75,4140278,1.619219
5,신항,환적,11176479.25,6464959,1.728778


In [83]:
grp_plot = grp_plot[grp_plot['청코드'] != '감천']

grp_plot

,청코드,환적여부,물동량,개수,TEU_FACTOR
2,북항,수출입(일반),4197997.25,2859718,1.467976
3,북항,환적,2314512.75,1569611,1.474577
4,신항,수출입(일반),6704016.75,4140278,1.619219
5,신항,환적,11176479.25,6464959,1.728778


In [84]:
fig = px.bar(
    grp_plot,
    x='청코드',
    y='TEU_FACTOR',
    color='환적여부',
    barmode='group',
    title='청코드, 환적여부별 컨테이너당 TEU'
)
fig.show()

In [98]:
grp2_plot = grp2.reset_index()

grp2_plot['환적여부'] = grp2_plot['환적여부'].map({
    True: '환적',
    False: '수출입(일반)'
})

grp2_plot

적공구분,청코드,환적여부,공컨,적컨,공컨비율(%)
0,감천,수출입(일반),1083.0,1739.00,38.38
1,감천,환적,0.0,6192.00,0.00
2,북항,수출입(일반),1314340.5,2883656.75,31.31
3,북항,환적,45790.0,2268722.75,1.98
4,신항,수출입(일반),1983822.0,4720194.75,29.59
5,신항,환적,713056.0,10463423.25,6.38


In [99]:
grp2_plot[grp2_plot['청코드'] != '감천']

적공구분,청코드,환적여부,공컨,적컨,공컨비율(%)
2,북항,수출입(일반),1314340.5,2883656.75,31.31
3,북항,환적,45790.0,2268722.75,1.98
4,신항,수출입(일반),1983822.0,4720194.75,29.59
5,신항,환적,713056.0,10463423.25,6.38


In [101]:
fig = px.bar(
    grp2_plot,
    x='청코드',
    y='공컨비율(%)',
    color='환적여부',
    barmode='group',
    title='청코드, 환적 여부별 공컨비율'
)

fig.show()

### 물동량이 몰리는 달에도 우리가 구한 TEU FACTOR와 같은 처리 효율 지표가 안정적으로 유지된다면, 해당 항만은 물동량 변화에 잘 대응하는 여유있는 운영능력을 갖추었다라고 판단할 수 있는 것이고, 반대로 물동량이 늘었을때 효율이 떨어진다면 처리능력에 병목이 있다 라고 볼 수 있다.

In [102]:
df

,연도,월,분기,청코드,내외항구분,수출입구분명,시설코드,시설명,부두구분명,아외국구분,적공구분,컨테이너수(10피트),컨테이너수(20피트),컨테이너수(40피트),컨테이너수(기타),전체개수,전체물동량,환적여부
0,2024,3,1,신항,외항,수입,6,신항 W 정박지,일반부두,외국선,적컨,0,3,8,0,11,19.00,False
1,2024,4,2,신항,외항,수입,6,신항 W 정박지,일반부두,아국선,적컨,0,9,3,0,12,15.00,False
2,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,공컨,0,1,0,0,1,1.00,True
3,2024,11,4,신항,외항,수입,7,신항 U 정박지,일반부두,아국선,공컨,0,57,2,0,59,61.00,False
4,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,적컨,0,31,100,0,131,231.00,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2404,2024,7,3,북항,외항,수입환적,8,자성대 부두,컨테이너부두,외국선,적컨,0,8769,6434,0,15203,21637.00,True
2405,2024,7,3,북항,외항,수출,8,자성대 부두,컨테이너부두,외국선,적컨,0,7719,6438,2,14159,20599.50,False
2406,2024,7,3,북항,외항,수입,8,자성대 부두,컨테이너부두,외국선,적컨,0,7818,7253,0,15071,22324.00,False
2407,2024,11,4,북항,외항,수출,8,자성대 부두,컨테이너부두,외국선,적컨,0,2519,2018,3,4540,6561.75,False


In [127]:
shinhang = df[df['청코드'] == '신항']

monthly_sin = shinhang.groupby('월').agg(물동량 = ('전체물동량', 'sum'), 개수=('전체개수', 'sum'))

monthly_sin = monthly_sin.reset_index()
monthly_sin['TEU_FACTOR'] = monthly_sin['물동량'] / monthly_sin['개수']

monthly_sin

,월,물동량,개수,TEU_FACTOR
0,1,1416542.50,846609,1.673196
1,2,1356151.25,803971,1.686816
2,3,1563893.50,932734,1.676677
3,4,1492720.75,891671,1.674071
4,5,1554553.00,920312,1.689159
5,6,1531476.50,912843,1.677700
6,7,1558413.50,922562,1.689224
7,8,1526833.50,900600,1.695351
8,9,1382264.25,815262,1.695485
9,10,1518211.25,896472,1.693540


In [129]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots



fig_dual = make_subplots(specs=[[{'secondary_y': True}]])# 보조 y축 하나 추가한디

fig_dual.add_trace(
    go.Bar(
        x=monthly_sin['월'], y=monthly_sin['물동량'], name='물동량(TEU)'
    ),
    secondary_y=False
)

fig_dual.add_trace(
    go.Scatter(
        x=monthly_sin['월'], y=monthly_sin['TEU_FACTOR'], mode='lines+markers', name='TEU_FACTOR'
    ),
    secondary_y=True
)

fig_dual.update_yaxes(secondary_y=True, range=[1.5, 1.8])
fig_dual.show()

## kpi 설계
- (신항) 환적화물 TEU Factor - 리프트당 처리 용적, 크레인, 야드의 생산성이 높다
- 신항, 북항 TEU Factor 격차(%) - 두 항만에 대한 상대적 생산성 차이
- 환적화물 공컨비율의 신항 북항 대비격차 - 낮을수록 처리능력이 수익화물에 더 집중되어 있다

In [134]:
grp

물동량       개수  TEU_FACTOR
청코드 환적여부                                   
감천  False      2822.00     1516    1.861478
    True       6192.00     3210    1.928972
북항  False   4197997.25  2859718    1.467976
    True    2314512.75  1569611    1.474577
신항  False   6704016.75  4140278    1.619219
    True   11176479.25  6464959    1.728778

In [133]:
kpi_teu_factor_sinhang = grp.loc[('신항', True), 'TEU_FACTOR']
kpi_teu_factor_sinhang

np.float64(1.7287780556690306)

In [136]:
kpi_teu_factor_bukhang = grp.loc[('북항', True), 'TEU_FACTOR']
kpi_teu_factor_bukhang

np.float64(1.4745772997258557)

In [138]:
kpi_teu_factor_gap = (kpi_teu_factor_sinhang / kpi_teu_factor_bukhang - 1) * 100
kpi_teu_factor_gap

np.float64(17.23888981543622)

In [139]:
grp2

적공구분              공컨           적컨  공컨비율(%)
청코드 환적여부                                  
감천  False     1083.0      1739.00    38.38
    True         0.0      6192.00     0.00
북항  False  1314340.5   2883656.75    31.31
    True     45790.0   2268722.75     1.98
신항  False  1983822.0   4720194.75    29.59
    True    713056.0  10463423.25     6.38

In [140]:
kpi_empty_ratio_sinhang = grp2.loc[('신항', True), '공컨비율(%)']
kpi_empty_ratio_sinhang

np.float64(6.38)

In [141]:
kpi_empty_ratio_bukhang = grp2.loc[('북항', True), '공컨비율(%)']
kpi_empty_ratio_bukhang

np.float64(1.98)

In [ ]:
# 여기서 더 나아가서 상관계수 ,, 물동량이랑 teu factor